# Marketing A/B Test

## 1. Design the Experiment

### Hypothesis

The group of people that saw the PSA public service agreement would have a greater conversion rate than those who saw the ads.

H0: p = p
H1: p > p

H1 is the alternative hypothesis and is of the form that the probability a person in the treatment group who saw ads converted into a paid customer is higher to that of the control group who only saw the public service agreement.

### Import Packages

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

### Load Data

In [3]:
df = pd.read_csv('data/marketing_AB.csv')

df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Unnamed: 0     588101 non-null  int64
 1   user id        588101 non-null  int64
 2   test group     588101 non-null  str  
 3   converted      588101 non-null  bool 
 4   total ads      588101 non-null  int64
 5   most ads day   588101 non-null  str  
 6   most ads hour  588101 non-null  int64
dtypes: bool(1), int64(4), str(2)
memory usage: 27.5 MB


In [16]:
df['test group'].unique()

<StringArray>
['ad', 'psa']
Length: 2, dtype: str

### Power Analysis - Calculate sample size

In [9]:
len(df)

588101

In [52]:
minimum_detectable_effect = 0.03

minimum_group_size = sms.NormalIndPower().solve_power(
    effect_size = minimum_detectable_effect,
    power=0.8,
    alpha=0.05,
    ratio=1
)

minimum_group_size = ceil(minimum_group_size)

minimum_group_size

17442

In [53]:
session_counts = df['user id'].value_counts(ascending=False)
session_counts[session_counts > 1].count()

np.int64(0)

## 2. Sampling - Allocate Samples

In [54]:
control_sample = df[df['test group'] == 'ad'].sample(n=minimum_group_size, random_state=0)
treatment_sample = df[df['test group'] == 'psa'].sample(n=minimum_group_size, random_state=0)

len(control_sample), len(treatment_sample)

(17442, 17442)

In [55]:
ab_test = pd.concat([control_sample, treatment_sample], axis=0)
ab_test.reset_index(drop=True, inplace=True)
ab_test

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,460261,1538442,ad,False,12,Wednesday,13
1,215236,1584782,ad,False,8,Thursday,17
2,346796,1014713,ad,False,5,Monday,13
3,243504,1453946,ad,False,10,Monday,18
4,173552,1620742,ad,False,6,Wednesday,17
...,...,...,...,...,...,...,...
34879,510443,912030,psa,False,20,Saturday,12
34880,156420,917521,psa,False,16,Thursday,8
34881,292672,905550,psa,False,7,Monday,9
34882,284448,904836,psa,False,16,Monday,21


## 3. Visualise Results - Calculate Metrics

In [60]:
conversion_rates = ab_test.groupby('test group')['converted']

std_prop = lambda x: np.std(x, ddof=0) # Standard deviation of the proportion
se_p = lambda x: stats.sem(x, ddof=0) # Standard error of the proportion (std / sqrt(n))

conversion_rates = conversion_rates.agg([np.mean, std_prop, se_p])
conversion_rates.columns = ['conversion_rate', 'std_deviation', 'std_error']

conversion_rates.style.format('{:.3f}')

,conversion_rate,std_deviation,std_error
test group,,,
ad,0.025,0.156,0.001
psa,0.018,0.131,0.001


## 4. Test the Hypothesis

In [65]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# 1. Isolate the target binary column
control_results = ab_test[ab_test['test group'] == 'ad']['converted']
treatment_results = ab_test[ab_test['test group'] == 'psa']['converted']

# 2. Compute proper counts
n_con = control_results.count()
n_treat = treatment_results.count()

successes = [control_results.sum(), treatment_results.sum()]
n_values = [n_con, n_treat]

# 3. Perform TWO-sample, ONE-tailed Z-test (H1: p_ad > p_psa)
z_stat, p_val = proportions_ztest(successes, nobs=n_values, alternative='larger')

# 4. Optional: Confidence Intervals (95%)
lower_con, upper_con = proportion_confint(successes[0], n_values[0], alpha=0.05)
lower_treat, upper_treat = proportion_confint(successes[1], n_values[1], alpha=0.05)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_val:.4f}")
print(f"Control 95% CI: [{lower_con:.4f}, {upper_con:.4f}]")
print(f"Treatment 95% CI: [{lower_treat:.4f}, {upper_treat:.4f}]")

Z-statistic: 4.7158
P-value: 0.0000
Control 95% CI: [0.0226, 0.0272]
Treatment 95% CI: [0.0156, 0.0196]


In [66]:
if p_val < 0.05:
    print('Reject the null hypothesis.')
else:
    print('Do not reject the null hypothesis.')

Reject the null hypothesis.
